In [1]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

from sklearn.metrics.pairwise import euclidean_distances


import json

from mistralai import Mistral

from pathlib import Path

# Preliminari

In [2]:
# Configurazione Mistral
client = Mistral(
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent

MODEL = constants.MISTRAL_EMBED
RESULTS_FILE_NAME = 'mistral_embeddings.jsonl'

# Load Data

In [3]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

train_data, validation_data, test_data = data['train'], data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(train_data.id) & set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")

len(train_data) = 187
len(validation_data) = 63
len(test_data) = 65


# Generazione

In [4]:
# Funzione generatrice
def generate_embedding(model: str, text: str | list[str]):
    response = client.embeddings.create(
        model=model,
        inputs=text
    )
    return response

In [5]:
# Esempio
if True:
    embeddings = generate_embedding(MODEL, ['Avengers... Uniti!', "Thor è l'avenger più forte!", 'La cina è un paese enorme!'])

In [6]:
embeddings

EmbeddingResponse(id='c438363008314f059107797d3b08f03b', object='list', model='mistral-embed-2312', usage=UsageInfo(prompt_tokens=32, completion_tokens=0, total_tokens=32, prompt_audio_seconds=None, request_count=None, prompt_tokens_details=None, prompt_token_details=None), data=[EmbeddingResponseData(object='embedding', embedding=[0.0102081298828125, 0.0187225341796875, 0.052337646484375, 0.00772857666015625, 0.026397705078125, 0.04241943359375, 0.0011281967163085938, -0.0069122314453125, -0.00345611572265625, -0.0430908203125, 0.020416259765625, 0.056182861328125, -0.0251617431640625, -0.007671356201171875, -0.02166748046875, 0.030242919921875, 0.0015306472778320312, 0.0005040168762207031, 0.0295562744140625, 0.03045654296875, 0.0102081298828125, -0.01241302490234375, -0.056854248046875, -0.00919342041015625, -0.00482177734375, 0.0165863037109375, -0.0209808349609375, -0.057098388671875, -0.054168701171875, -0.017822265625, -0.00148773193359375, -0.039276123046875, 0.022674560546875,

In [7]:
x = generate_embedding(MODEL, 'Avengers... Uniti!')

In [9]:
len(x.data[0].embedding)

1024

In [17]:
if True: # To see the full output
    pprint(embeddings.data)
    print(embeddings.data[0].embedding)

[EmbeddingResponseData(object='embedding', embedding=[0.0102081298828125, 0.0187225341796875, 0.052337646484375, 0.00772857666015625, 0.026397705078125, 0.04241943359375, 0.0011281967163085938, -0.0069122314453125, -0.00345611572265625, -0.0430908203125, 0.020416259765625, 0.056182861328125, -0.0251617431640625, -0.007671356201171875, -0.02166748046875, 0.030242919921875, 0.0015306472778320312, 0.0005040168762207031, 0.0295562744140625, 0.03045654296875, 0.0102081298828125, -0.01241302490234375, -0.056854248046875, -0.00919342041015625, -0.00482177734375, 0.0165863037109375, -0.0209808349609375, -0.057098388671875, -0.054168701171875, -0.017822265625, -0.00148773193359375, -0.039276123046875, 0.022674560546875, -0.006683349609375, 0.03814697265625, 0.01410675048828125, -0.016021728515625, -0.04827880859375, 0.0552978515625, -0.042877197265625, 0.00662994384765625, -0.032257080078125, 0.0059814453125, -0.0217742919921875, 0.0011425018310546875, -0.036773681640625, -0.0014104843139648438

In [18]:
print(euclidean_distances([embeddings.data[0].embedding], [embeddings.data[1].embedding]))
print(euclidean_distances([embeddings.data[0].embedding], [embeddings.data[2].embedding]))
print(euclidean_distances([embeddings.data[1].embedding], [embeddings.data[2].embedding]))

[[0.63812687]]
[[0.82143263]]
[[0.73572005]]


## EMBEDDINGS

In [19]:
print(MODEL)
df = pd.concat([train_data, validation_data, test_data], ignore_index=True)
ids = []
splits = []
embeddings = []
for i, row in df.iterrows():
    row = df.iloc[i]
    embedding = generate_embedding(MODEL, row[constants.REPORT_COLUMN_NAME])
    embeddings.append(embedding.data[0].embedding)
    splits.append(row['split'])
    ids.append(int(row[constants.ID_COMULM_NAME]))

mistral-embed-2312


In [22]:
assert len(ids) == len(splits) == len(embeddings)

In [23]:
results_dicts = []
assert len(ids) == len(splits) == len(embeddings)
for id, split, embedding, in zip(ids, splits, embeddings):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': id,
            'embedding': embedding
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'embeddings' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')